# 03 - Baseline & Basic Modeling
## Building Energy Efficiency: Heating & Cooling Load Prediction
**DEPI Graduation Project - Team CloudXAi (NHA-5-165)**

This notebook trains, evaluates, and compares baseline models:
1. Linear Regression (Multivariate Baseline)
2. Decision Tree Regressor
3. Random Forest Regressor


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

from src.energy_efficiency.config import TARGETS, RANDOM_STATE
from src.energy_efficiency.data_loader import load_processed_data
from src.energy_efficiency.preprocessing import build_preprocessor
from src.energy_efficiency.evaluation import evaluate_model

print("Modeling environment ready.")


### 1. Load Processed Datasets


In [ ]:
train_df, test_df = load_processed_data()
features_cols = [c for c in train_df.columns if c not in TARGETS]
X_train, y_train = train_df[features_cols], train_df[TARGETS]
X_test, y_test = test_df[features_cols], test_df[TARGETS]

prep_linear = build_preprocessor(scale=True, onehot=True, engineer=True).fit(X_train)
prep_tree = build_preprocessor(scale=False, onehot=False, engineer=True).fit(X_train)


### 2. Model Training & Evaluation


In [ ]:
models = {
    "Linear Regression": (LinearRegression(), prep_linear.transform(X_train), prep_linear.transform(X_test)),
    "Decision Tree": (DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE), prep_tree.transform(X_train), prep_tree.transform(X_test)),
    "Random Forest": (RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1), prep_tree.transform(X_train), prep_tree.transform(X_test)),
}

results = []
predictions = {}

for name, (model, tr_X, te_X) in models.items():
    model.fit(tr_X, y_train)
    preds = model.predict(te_X)
    predictions[name] = preds

    for i, tgt in enumerate(TARGETS):
        results.append(evaluate_model(y_test[tgt], preds[:, i], tgt, name))

results_df = pd.DataFrame(results)
display(results_df)


### 3. Actual vs Predicted Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_preds = predictions["Random Forest"]

for i, tgt in enumerate(TARGETS):
    axes[i].scatter(y_test[tgt], rf_preds[:, i], alpha=0.7, color="teal", edgecolors="black", linewidth=0.5)
    lims = [y_test[tgt].min(), y_test[tgt].max()]
    axes[i].plot(lims, lims, "r--", linewidth=1.5, label="Perfect Fit (y = x)")
    axes[i].set_title(f"Random Forest - {tgt} (R² = {r2_score(y_test[tgt], rf_preds[:, i]):.4f})")
    axes[i].set_xlabel("Actual Load")
    axes[i].set_ylabel("Predicted Load")
    axes[i].legend()

plt.tight_layout()
plt.show()


### 4. Key Findings
- Linear Regression achieves an $R^2$ of ~0.92 for heating and ~0.90 for cooling, demonstrating that linear relationships capture a large fraction of the variance.
- Random Forest captures non-linear geometric interactions, boosting $R^2$ to **0.9977** on Heating Load and **0.9609** on Cooling Load.
